In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################


username = "aacuser"
password = "amgborro"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)


image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())


app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
   
    html.Center([
        html.Img(
            src='data:image/png;base64,{}'.format(
                encoded_image.decode()
            ),
            style={'width': '200px'}
        ),
        html.H1('Grazioso Salvare Rescue Dashboard'),
        html.H3('Ashlynn Borro')
    ]),
    html.Hr(),
    html.Div(

        
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'Water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'Disaster'},
                {'label': 'Reset', 'value': 'Reset'}
            ],
            value='Reset',
            labelStyle={'display': 'inline-block', 'marginRight': '20px'}
        ),

    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                row_selectable='single',
                columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                data=df.to_dict('records'),
 

                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        
        html.Div(
            id='map-id',
            className='col s12 m6',
            style={
                'width': '50%',
                'minHeight': '500px'
            }
        )

        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
        if filter_type == 'Reset':
            return df.to_dict('records')
        
        elif filter_type == 'Water':
            query = {
                "animal_type": "Dog",
                "breed": {"$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]},
                "sex_upon_outcome": "Intact Female",
                "age_upon_outcome_in_weeks": {
                    "$gte": 26,
                    "$lte": 156
                }
            }

        elif filter_type == 'Mountain':
            query = {
                "animal_type": "Dog",
                "breed": {"$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]},
                "sex_upon_outcome": "Intact Male",
                "age_upon_outcome_in_weeks": {
                    "$gte": 26,
                    "$lte": 156
                }
            }

        elif filter_type == 'Disaster':
            query = {
                "animal_type": "Dog",
                "breed": {"$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]},
                "sex_upon_outcome": "Intact Male",
                "age_upon_outcome_in_weeks": {
                    "$gte": 20,
                    "$lte": 300
                }
            }

        else:
            return df.to_dict('records')

        filtered_df = pd.DataFrame.from_records(db.read(query))
        filtered_df.drop(columns=['_id'], inplace=True, errors='ignore')

        return filtered_df.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
        
        if not viewData:
            return [html.P("No animal records available for this filter.")]

        chart_df = pd.DataFrame(viewData)

        if 'breed' not in chart_df.columns:
            return [html.P("Breed information is unavailable.")]
    
        breed_counts = chart_df['breed'].dropna().value_counts().head(10)

        if breed_counts.empty:
            return [html.P("No breed data available.")]

        fig = px.pie(
            names=breed_counts.index,
            values=breed_counts.values,
            title='Top 10 Animal Breeds'
        )

        return [dcc.Graph(figure=fig)]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
   
    if not selected_columns:
        return []

    return [
        {
            'if': {'column_id': i},
            'background_color': '#D2F3FF'
        }
        for i in selected_columns
    ]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])

def update_map(viewData, index):
    # Default map location: Austin, Texas
    center = [30.75, -97.48]

    def create_map(children=None):
        return [
            dl.Map(
                style={'width': '100%', 'height': '500px'},
                center=center,
                zoom=10,
                children=[
                    dl.TileLayer(id='base-layer-id'),
                    *(children or [])
                ]
            )
        ]

    # Handle empty or missing data
    if not viewData:
        return create_map()

    dff = pd.DataFrame(viewData)

    # Handle missing selection or invalid row index
    if not index:
        row = 0
    else:
        row = index[0]

    if not isinstance(row, int) or row < 0 or row >= len(dff):
        return create_map()

    animal = dff.iloc[row]

    # Check that location fields exist
    if ('location_lat' not in animal.index or
            'location_long' not in animal.index):
        return create_map()

    # Convert coordinates to numbers
    lat = pd.to_numeric(animal['location_lat'], errors='coerce')
    lon = pd.to_numeric(animal['location_long'], errors='coerce')

    # Handle missing or invalid coordinates
    if pd.isna(lat) or pd.isna(lon):
        return create_map()

    if not (-90 <= lat <= 90 and -180 <= lon <= 180):
        return create_map()

    # Retrieve animal information safely
    breed = animal.get('breed')
    name = animal.get('name')

    breed = str(breed) if pd.notna(breed) else 'Unknown Breed'
    name = str(name) if pd.notna(name) else 'Unknown Name'

    # Create marker with animal information
    marker = dl.Marker(
        position=[float(lat), float(lon)],
        children=[
            dl.Tooltip(breed),
            dl.Popup([
                html.H4("Animal Name"),
                html.P(name)
            ])
        ]
    )

    return create_map([marker])

# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://michaelsleep-storybook-3000.codio.io/proxy/8050/
